# Post-clustering filters: synthesis datasets, spectral entropy and confidence-aware molecular networks

1. *Calculate deltas pre-/post-clustering*: number of scans and unique delta masses after removing synthesis datasets (a cluster is kept only if ≥ 2 of its scans come from non-synthesis datasets).
2. *Entropy filter*: remove clusters with spectral entropy > 7.5 (> 7.0 for the C26 aceto core).
3. *Molecular networking filter with SpecReBoot*: read the SpecReBoot GraphML networks; remove a whole molecular family when > 1/3 of its annotated nodes match non-bile acids, otherwise only those nodes; singletons are removed only when annotated as non-bile acid.
4. Keep the scans that survive both filters (C28 1OH: entropy only) and write the final MGFs.
5. *Deltas calculation*: scans and unique delta masses per core in the final MGFs.

In [ ]:
import os

# Root of the local project folder (the one holding Falcon/, Queries_final_mgf/,
# final_mgfs_library_generation/, ...). Edit this single line to run elsewhere.
PROJECT_DIR = '/media/dorrestein/Helena2/Bile_acids_update'

In [ ]:
import os
import warnings

import pandas as pd

warnings.filterwarnings('ignore')

In [ ]:
falcon_mgf_files = {
    "C24_nonhydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_nonhydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C24_monohydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_monohydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C24_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_dihydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C24_trihydroxy_part1": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_final_trihydroxy_part1_falcon_eps01_minsamples2_scan.mgf",
    "C24_trihydroxy_part2": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_final_trihydroxy_part2_falcon_eps01_minsamples2_scan.mgf",
    "C24_trihydroxy_part3": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_final_trihydroxy_part3_falcon_eps01_minsamples2_scan.mgf",
    "C24_tetrahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_tetrahydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C24_pentahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_pentahydroxy_falcon_eps01_minsamples2_scan.mgf",

    "C23_monohydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C23_monohydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C23_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C23_dihydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C23_trihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C23_trihydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C23_tetrahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C23_tetrahydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C23_pentahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C23_pentahydroxy_falcon_eps01_minsamples2_scan.mgf",

    "C27_monohydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C27_monohydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C27_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C27_dihydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C27_trihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C27_trihydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C27_tetrahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C27_tetrahydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C27_pentahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C27_pentahydroxy_falcon_eps01_minsamples2_scan.mgf",

    "Aceto_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/aceto_dihydroxy_falcon_eps01_minsamples2_scan.mgf",

    "C24_alkylamine_monohydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_alkylamine_monohydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C24_alkylamine_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_alkylamine_dihydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C24_alkylamine_trihydroxy_part1": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part1_falcon_eps01_minsamples2_scan.mgf",
    "C24_alkylamine_trihydroxy_part2": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part2_falcon_eps01_minsamples2_scan.mgf",
    "C24_alkylamine_trihydroxy_part3": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part3_falcon_eps01_minsamples2_scan.mgf",
    "C24_alkylamine_trihydroxy_part4": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part4_falcon_eps01_minsamples2_scan.mgf",
    "C24_alkylamine_tetrahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_alkylamine_tetrahydroxy_falcon_eps01_minsamples2_scan.mgf",
    "C24_alkylamine_pentahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_alkylamine_pentahydroxy_falcon_eps01_minsamples2_scan.mgf",

    "C28_alcohol_1OH": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C28_alcohol_1OH_falcon_eps01_minsamples2_scan.mgf",
    "C28_alcohol_2OH": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C28_alcohol_2OH_falcon_eps01_minsamples2_scan.mgf",
    "C28_alcohol_3OH": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C28_alcohol_3OH_falcon_eps01_minsamples2_scan.mgf",
}

falcon_csv_files = {
    "C24_nonhydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_nonhydroxy_falcon_eps01_minsamples2.csv",
    "C24_monohydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_monohydroxy_falcon_eps01_minsamples2.csv",
    "C24_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_dihydroxy_falcon_eps01_minsamples2.csv",
    "C24_trihydroxy_part1": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_final_trihydroxy_part1_falcon_eps01_minsamples2.csv",
    "C24_trihydroxy_part2": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_final_trihydroxy_part2_falcon_eps01_minsamples2.csv",
    "C24_trihydroxy_part3": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_final_trihydroxy_part3_falcon_eps01_minsamples2.csv",
    "C24_tetrahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_tetrahydroxy_falcon_eps01_minsamples2.csv",
    "C24_pentahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_pentahydroxy_falcon_eps01_minsamples2.csv",

    "C23_monohydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C23_monohydroxy_falcon_eps01_minsamples2.csv",
    "C23_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C23_dihydroxy_falcon_eps01_minsamples2.csv",
    "C23_trihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C23_trihydroxy_falcon_eps01_minsamples2.csv",
    "C23_tetrahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C23_tetrahydroxy_falcon_eps01_minsamples2.csv",
    "C23_pentahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C23_pentahydroxy_falcon_eps01_minsamples2.csv",

    "C27_monohydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C27_monohydroxy_falcon_eps01_minsamples2.csv",
    "C27_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C27_dihydroxy_falcon_eps01_minsamples2.csv",
    "C27_trihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C27_trihydroxy_falcon_eps01_minsamples2.csv",
    "C27_tetrahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C27_tetrahydroxy_falcon_eps01_minsamples2.csv",
    "C27_pentahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C27_pentahydroxy_falcon_eps01_minsamples2.csv",

    "Aceto_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/aceto_dihydroxy_falcon_eps01_minsamples2.csv",

    "C24_alkylamine_monohydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_monohydroxy_falcon_eps01_minsamples2.csv",
    "C24_alkylamine_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_dihydroxy_falcon_eps01_minsamples2.csv",
    "C24_alkylamine_trihydroxy_part1": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part1_falcon_eps01_minsamples2.csv",
    "C24_alkylamine_trihydroxy_part2": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part2_falcon_eps01_minsamples2.csv",
    "C24_alkylamine_trihydroxy_part3": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part3_falcon_eps01_minsamples2.csv",
    "C24_alkylamine_trihydroxy_part4": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part4_falcon_eps01_minsamples2.csv",
    "C24_alkylamine_tetrahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_tetrahydroxy_falcon_eps01_minsamples2.csv",
    "C24_alkylamine_pentahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_pentahydroxy_falcon_eps01_minsamples2.csv",

    "C28_alcohol_1OH": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C28_alcohol_1OH_falcon_eps01_minsamples2.csv",
    "C28_alcohol_2OH": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C28_alcohol_2OH_falcon_eps01_minsamples2.csv",
    "C28_alcohol_3OH": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C28_alcohol_3OH_falcon_eps01_minsamples2.csv",
}

annotation_table_pre_falcon = {
    "C24_nonhydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_final/Nonhydroxy/download_mgfs/final_mgf/C24_Nonhydroxy_input_falcon_charge_annotation_table.tsv",
    "C24_monohydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_final/Monohydroxy/download_mgfs/final_mgf/C24_Monohydroxy_input_falcon_charge_annotation_table.tsv",
    "C24_dihydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_final/Dihydroxy/download_mgfs/final_mgf/C24_dihydroxy_input_falcon_charge_annotation_table.tsv",
    "C24_trihydroxy_part1": f"{PROJECT_DIR}/Queries_final_mgf/C24_final/Trihydroxy/download_mgfs/final_mgf/C24_Trihydroxy_input_falcon_charge_part1_annotation_table.tsv",
    "C24_trihydroxy_part2": f"{PROJECT_DIR}/Queries_final_mgf/C24_final/Trihydroxy/download_mgfs/final_mgf/C24_Trihydroxy_input_falcon_charge_part2_annotation_table.tsv",
    "C24_trihydroxy_part3": f"{PROJECT_DIR}/Queries_final_mgf/C24_final/Trihydroxy/download_mgfs/final_mgf/C24_Trihydroxy_input_falcon_charge_part3_annotation_table.tsv",
    "C24_tetrahydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_final/Tetrahydroxy/download_mgfs/final_mgf/C24_Tetrahydroxy_input_falcon_charge_annotation_table.tsv",
    "C24_pentahydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_final/Pentahydroxy/download_mgfs/final_mgf/C24_Pentahydroxy_input_falcon_charge_annotation_table.tsv",

    "C23_monohydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C23/C23_monohydroxy/download_mgfs/final_mgf/C23_Monohydroxy_input_falcon_charge_annotation_table.tsv",
    "C23_dihydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C23/C23_dihydroxy/download_mgfs/final_mgf/C23_dihydroxy_input_falcon_charge_annotation_table.tsv",
    "C23_trihydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C23/C23_trihydroxy/download_mgfs/final_mgf/C23_trihydroxy_input_falcon_charge_annotation_table.tsv",
    "C23_tetrahydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C23/C23_tetrahydroxy/download_mgfs/final_mgf/C23_tetrahydroxy_input_falcon_charge_annotation_table.tsv",
    "C23_pentahydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C23/C23_pentahydroxy/download_mgfs/final_mgf/C23_pentahydroxy_input_falcon_charge_annotation_table.tsv",

    "C27_monohydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C27/C27_monohydroxy/download_mgfs/final_mgf/C27_monohydroxy_input_falcon_charge_annotation_table.tsv",
    "C27_dihydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C27/C27_dihydroxy/download_mgfs/final_mgf/C27_dihydroxy_input_falcon_charge_annotation_table.tsv",
    "C27_trihydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C27/C27_trihydroxy/download_mgfs/final_mgf/C27_trihydroxy_input_falcon_charge_annotation_table.tsv",
    "C27_tetrahydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C27/C27_tetrahydroxy/download_mgfs/final_mgf/C27_tetrahydroxy_input_falcon_charge_annotation_table.tsv",
    "C27_pentahydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C27/C27_pentahydroxy/download_mgfs/final_mgf/C27_pentahydroxy_input_falcon_charge_annotation_table.tsv",

    "Aceto_dihydroxy": f"{PROJECT_DIR}/Queries_final_mgf/Aceto/Dihydroxy/download_mgfs/final_mgf/Aceto_dihydroxy_input_falcon_charge_renumbered_annotation_table.tsv",

    "C24_alkylamine_monohydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_monohydroxy/download_mgfs/final_mgf/C24_alkylamine_monohydroxy_input_falcon_charge_annotation_table.tsv",
    "C24_alkylamine_dihydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_dihydroxy/download_mgfs/final_mgf/C24_alkylamine_dihydroxy_input_falcon_charge_annotation_table.tsv",
    "C24_alkylamine_trihydroxy_part1": f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part1_annotation_table.tsv",
    "C24_alkylamine_trihydroxy_part2": f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part2_annotation_table.tsv",
    "C24_alkylamine_trihydroxy_part3": f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part3_annotation_table.tsv",
    "C24_alkylamine_trihydroxy_part4": f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part4_annotation_table.tsv",
    "C24_alkylamine_tetrahydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_tetrahydroxy/download_mgfs/final_mgf/C24_alkylamine_tetrahydroxy_input_falcon_charge_annotation_table.tsv",
    "C24_alkylamine_pentahydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_pentahydroxy/download_mgfs/final_mgf/C24_alkylamine_pentahydroxy_input_falcon_charge_annotation_table.tsv",

    "C28_alcohol_1OH": f"{PROJECT_DIR}/Queries_final_mgf/C28_alcohol/One_OH/download_mgfs/final_mgf/C28_alcohol_1OH_input_falcon_351_C28O1_charge_annotation_table.tsv",
    "C28_alcohol_2OH": f"{PROJECT_DIR}/Queries_final_mgf/C28_alcohol/Two_OH/download_mgfs/final_mgf/C28_alcohol_2OH_input_falcon_367_C28O2_charge_annotation_table.tsv",
    "C28_alcohol_3OH": f"{PROJECT_DIR}/Queries_final_mgf/C28_alcohol/Three_OH/download_mgfs/final_mgf/C28_alcohol_3OH_input_falcon_383_C28O3_charge_annotation_table.tsv",
}

deltas_calculation = {
    "C24_nonhydroxy": 361.3101,
    "C24_monohydroxy": 377.305,
    "C24_dihydroxy": 393.2999,
    "C24_trihydroxy_part1": 409.2949,
    "C24_trihydroxy_part2": 409.2949,
    "C24_trihydroxy_part3": 409.2949,
    "C24_tetrahydroxy": 425.2898,
    "C24_pentahydroxy": 441.2847,

    "C23_monohydroxy": 363.2894,
    "C23_dihydroxy": 379.2843,
    "C23_trihydroxy": 395.2792,
    "C23_tetrahydroxy": 411.2741,
    "C23_pentahydroxy": 427.269,

    "C27_monohydroxy": 419.352,
    "C27_dihydroxy": 435.3469,
    "C27_trihydroxy": 451.3418,
    "C27_tetrahydroxy": 467.3367,
    "C27_pentahydroxy": 483.3316,

    "Aceto_dihydroxy": 435.3105,

    "C24_alkylamine_monohydroxy": 377.305,
    "C24_alkylamine_dihydroxy": 393.2999,
    "C24_alkylamine_trihydroxy_part1": 409.2949,
    "C24_alkylamine_trihydroxy_part2": 409.2949,
    "C24_alkylamine_trihydroxy_part3": 409.2949,
    "C24_alkylamine_trihydroxy_part4": 409.2949,
    "C24_alkylamine_tetrahydroxy": 425.2898,
    "C24_alkylamine_pentahydroxy": 441.2847,
}

In [ ]:
# it has to be at least two scans not from synthesis project.
synthesis = [
    "MSV000099509",
    "MSV000100947",
    "MSV000100886",
    "MSV000100534",
    "MSV000100532",
    "MSV000100441",
    "MSV000100347",
    "MSV000100095",
    "MSV000099376",
    "MSV000099375",
    "MSV000099242",
    "MSV000099090",
    "MSV000098640",
    "MSV000098639",
    "MSV000098637",
    "MSV000098628",
    "MSV000097885",
    "MSV000097869",
    "MSV000095820",
    "MSV000094447",
    "MSV000094393",
    "MSV000094391",
]

### Calculate deltas pre-clustering

In [ ]:
# annotation tables pre-falcon come from the annotation_table_pre_falcon dictionary
summary_rows = []

for table_name, table_path in annotation_table_pre_falcon.items():
    annotation_table = pd.read_csv(table_path, sep='\t')

    # Make sure no unnecessary spaces in column names.
    annotation_table.columns = annotation_table.columns.str.strip()

    # Create dataset column and remove scans from synthesis project.
    annotation_table['dataset'] = annotation_table['original_path'].str.split('/').str[2]
    annotation_table = annotation_table[~annotation_table['dataset'].isin(synthesis)]

    n_scans = len(annotation_table)
    print(f"{table_name}: number of scans after synthesis filter, pre-cluster: {n_scans}")

    delta_to_subtract = deltas_calculation.get(table_name)

    if delta_to_subtract is None or pd.isna(delta_to_subtract):
        n_deltas = "unspecified"
    else:
        annotation_table['delta'] = annotation_table['pepmass'] - delta_to_subtract
        annotation_table['delta_rounded'] = annotation_table['delta'].round(2)
        n_deltas = annotation_table['delta_rounded'].nunique()

    print(f"{table_name}: number of deltas after synthesis filter, pre-cluster: {n_deltas}")
    print('-' * 100)

    summary_rows.append({
        'table_name': table_name,
        'n_scans_pre_cluster': n_scans,
        'n_deltas_pre_cluster': n_deltas,
    })

summary_df = pd.DataFrame(summary_rows)
summary_outfile = f'{PROJECT_DIR}/Scripts_data_analysis/deltas_calculation/deltas_preclustering_summary.tsv'
summary_df.to_csv(summary_outfile, sep='\t', index=False)

print(f"Summary table exported to: {summary_outfile}")
summary_df

### Calculate deltas post-clustering

In [ ]:
import os

# Post-clustering summary for all tables
summary_rows_post = []
synthesis_set = set(synthesis)
base_dir = PROJECT_DIR


def resolve_path(path):
    return path if os.path.isabs(path) else os.path.join(base_dir, path)


def keep_cluster(group):
    in_list = group['dataset'].isin(synthesis_set)
    not_in_list = ~in_list

    # If all datasets are from the synthesis list, remove.
    if in_list.all():
        return False

    # If mixed, keep only if at least 2 rows are not in the list.
    if not_in_list.sum() >= 2:
        return True

    return False


for table_name, csv_path in falcon_csv_files.items():
    # Import the CSV file exported from falcon.
    df = pd.read_csv(resolve_path(csv_path), sep=',', skiprows=25)

    # Add 1 to cluster and keep only assigned clusters.
    df['cluster'] = df['cluster'] + 1
    df = df[df['cluster'] > 0]

    # Create scan column to match annotation table.
    df['scan_combined_mgf'] = df['identifier'].str.split(':').str[-1].astype(int)

    # Import annotation table pre-falcon for the same table.
    annotation_table = pd.read_csv(resolve_path(annotation_table_pre_falcon[table_name]), sep='\t')
    annotation_table.columns = annotation_table.columns.str.strip()

    # Merge and derive dataset source.
    merged_df = pd.merge(df, annotation_table, on='scan_combined_mgf', how='left')
    merged_df['dataset'] = merged_df['original_path'].str.split('/').str[2]

    # Filter clusters based on synthesis rule.
    merged_df = merged_df.groupby('cluster').filter(keep_cluster)

    n_scans_post = merged_df['cluster'].nunique()
    print(f"{table_name}: number of scans after synthesis filter, post-cluster: {n_scans_post}")

    delta_to_subtract = deltas_calculation.get(table_name)

    if delta_to_subtract is None or pd.isna(delta_to_subtract):
        n_deltas_post = 'unspecified'
    else:
        merged_df['delta'] = merged_df['pepmass'] - delta_to_subtract
        merged_df['delta_rounded'] = merged_df['delta'].round(2)
        n_deltas_post = merged_df['delta_rounded'].nunique()

    print(f"{table_name}: number of deltas after synthesis filter, post-cluster: {n_deltas_post}")
    print('-' * 100)

    summary_rows_post.append({
        'table_name': table_name,
        'n_scans_post_cluster': n_scans_post,
        'n_deltas_post_cluster': n_deltas_post,
    })

summary_post_df = pd.DataFrame(summary_rows_post)
summary_post_outfile = f'{PROJECT_DIR}/Scripts_data_analysis/deltas_calculation/deltas_postclustering_summary.tsv'
summary_post_df.to_csv(summary_post_outfile, sep='\t', index=False)

print(f"Post-clustering summary table exported to: {summary_post_outfile}")
summary_post_df

### Entropy filter

In [ ]:
falcon_csv_files = {
    "C24_nonhydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_nonhydroxy_falcon_eps01_minsamples2.csv",
    "C24_monohydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_monohydroxy_falcon_eps01_minsamples2.csv",
    "C24_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_dihydroxy_falcon_eps01_minsamples2.csv",
    "C24_trihydroxy_part1": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_final_trihydroxy_part1_falcon_eps01_minsamples2.csv",
    "C24_trihydroxy_part2": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_final_trihydroxy_part2_falcon_eps01_minsamples2.csv",
    "C24_trihydroxy_part3": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_final_trihydroxy_part3_falcon_eps01_minsamples2.csv",
    "C24_tetrahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_tetrahydroxy_falcon_eps01_minsamples2.csv",
    "C24_pentahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_pentahydroxy_falcon_eps01_minsamples2.csv",

    "C23_monohydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C23_monohydroxy_falcon_eps01_minsamples2.csv",
    "C23_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C23_dihydroxy_falcon_eps01_minsamples2.csv",
    "C23_trihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C23_trihydroxy_falcon_eps01_minsamples2.csv",
    "C23_tetrahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C23_tetrahydroxy_falcon_eps01_minsamples2.csv",
    "C23_pentahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C23_pentahydroxy_falcon_eps01_minsamples2.csv",

    "C27_monohydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C27_monohydroxy_falcon_eps01_minsamples2.csv",
    "C27_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C27_dihydroxy_falcon_eps01_minsamples2.csv",
    "C27_trihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C27_trihydroxy_falcon_eps01_minsamples2.csv",
    "C27_tetrahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C27_tetrahydroxy_falcon_eps01_minsamples2.csv",
    "C27_pentahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C27_pentahydroxy_falcon_eps01_minsamples2.csv",

    "Aceto_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/aceto_dihydroxy_falcon_eps01_minsamples2.csv",

    "C24_alkylamine_monohydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_monohydroxy_falcon_eps01_minsamples2.csv",
    "C24_alkylamine_dihydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_dihydroxy_falcon_eps01_minsamples2.csv",
    "C24_alkylamine_trihydroxy_part1": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part1_falcon_eps01_minsamples2.csv",
    "C24_alkylamine_trihydroxy_part2": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part2_falcon_eps01_minsamples2.csv",
    "C24_alkylamine_trihydroxy_part3": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part3_falcon_eps01_minsamples2.csv",
    "C24_alkylamine_trihydroxy_part4": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_part4_falcon_eps01_minsamples2.csv",
    "C24_alkylamine_tetrahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_tetrahydroxy_falcon_eps01_minsamples2.csv",
    "C24_alkylamine_pentahydroxy": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_pentahydroxy_falcon_eps01_minsamples2.csv",

    "C28_alcohol_1OH": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C28_alcohol_1OH_falcon_eps01_minsamples2.csv",
    "C28_alcohol_2OH": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C28_alcohol_2OH_falcon_eps01_minsamples2.csv",
    "C28_alcohol_3OH": f"{PROJECT_DIR}/Falcon/all_falcon_outputs/C28_alcohol_3OH_falcon_eps01_minsamples2.csv",
}

annotation_table_pre_falcon = {
    "C24_nonhydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_final/Nonhydroxy/download_mgfs/final_mgf/C24_Nonhydroxy_input_falcon_charge_annotation_table.tsv",
    "C24_monohydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_final/Monohydroxy/download_mgfs/final_mgf/C24_Monohydroxy_input_falcon_charge_annotation_table.tsv",
    "C24_dihydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_final/Dihydroxy/download_mgfs/final_mgf/C24_dihydroxy_input_falcon_charge_annotation_table.tsv",
    "C24_trihydroxy_combined": f"{PROJECT_DIR}/Queries_final_mgf/C24_final/Trihydroxy/download_mgfs/final_mgf/C24_Trihydroxy_input_falcon_charge_part1_annotation_table.tsv",
    "C24_tetrahydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_final/Tetrahydroxy/download_mgfs/final_mgf/C24_Tetrahydroxy_input_falcon_charge_annotation_table.tsv",
    "C24_pentahydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_final/Pentahydroxy/download_mgfs/final_mgf/C24_Pentahydroxy_input_falcon_charge_annotation_table.tsv",

    "C23_monohydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C23/C23_monohydroxy/download_mgfs/final_mgf/C23_Monohydroxy_input_falcon_charge_annotation_table.tsv",
    "C23_dihydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C23/C23_dihydroxy/download_mgfs/final_mgf/C23_dihydroxy_input_falcon_charge_annotation_table.tsv",
    "C23_trihydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C23/C23_trihydroxy/download_mgfs/final_mgf/C23_trihydroxy_input_falcon_charge_annotation_table.tsv",
    "C23_tetrahydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C23/C23_tetrahydroxy/download_mgfs/final_mgf/C23_tetrahydroxy_input_falcon_charge_annotation_table.tsv",
    "C23_pentahydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C23/C23_pentahydroxy/download_mgfs/final_mgf/C23_pentahydroxy_input_falcon_charge_annotation_table.tsv",

    "C27_monohydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C27/C27_monohydroxy/download_mgfs/final_mgf/C27_monohydroxy_input_falcon_charge_annotation_table.tsv",
    "C27_dihydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C27/C27_dihydroxy/download_mgfs/final_mgf/C27_dihydroxy_input_falcon_charge_annotation_table.tsv",
    "C27_trihydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C27/C27_trihydroxy/download_mgfs/final_mgf/C27_trihydroxy_input_falcon_charge_annotation_table.tsv",
    "C27_tetrahydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C27/C27_tetrahydroxy/download_mgfs/final_mgf/C27_tetrahydroxy_input_falcon_charge_annotation_table.tsv",
    "C27_pentahydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C27/C27_pentahydroxy/download_mgfs/final_mgf/C27_pentahydroxy_input_falcon_charge_annotation_table.tsv",

    "Aceto_dihydroxy": f"{PROJECT_DIR}/Queries_final_mgf/Aceto/Dihydroxy/download_mgfs/final_mgf/Aceto_dihydroxy_input_falcon_charge_renumbered_annotation_table.tsv",

    "C24_alkylamine_monohydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_monohydroxy/download_mgfs/final_mgf/C24_alkylamine_monohydroxy_input_falcon_charge_annotation_table.tsv",
    "C24_alkylamine_dihydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_dihydroxy/download_mgfs/final_mgf/C24_alkylamine_dihydroxy_input_falcon_charge_annotation_table.tsv",
    "C24_alkylamine_trihydroxy_combined": f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part1_annotation_table.tsv",
    "C24_alkylamine_tetrahydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_tetrahydroxy/download_mgfs/final_mgf/C24_alkylamine_tetrahydroxy_input_falcon_charge_annotation_table.tsv",
    "C24_alkylamine_pentahydroxy": f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_pentahydroxy/download_mgfs/final_mgf/C24_alkylamine_pentahydroxy_input_falcon_charge_annotation_table.tsv",

    "C28_alcohol_1OH": f"{PROJECT_DIR}/Queries_final_mgf/C28_alcohol/One_OH/download_mgfs/final_mgf/C28_alcohol_1OH_input_falcon_351_C28O1_charge_annotation_table.tsv",
    "C28_alcohol_2OH": f"{PROJECT_DIR}/Queries_final_mgf/C28_alcohol/Two_OH/download_mgfs/final_mgf/C28_alcohol_2OH_input_falcon_367_C28O2_charge_annotation_table.tsv",
    "C28_alcohol_3OH": f"{PROJECT_DIR}/Queries_final_mgf/C28_alcohol/Three_OH/download_mgfs/final_mgf/C28_alcohol_3OH_input_falcon_383_C28O3_charge_annotation_table.tsv",
}

deltas_calculation = {
    "C24_nonhydroxy": 361.3101,
    "C24_monohydroxy": 377.305,
    "C24_dihydroxy": 393.2999,
    "C24_trihydroxy_combined": 409.2949,
    "C24_tetrahydroxy": 425.2898,
    "C24_pentahydroxy": 441.2847,

    "C23_monohydroxy": 363.2894,
    "C23_dihydroxy": 379.2843,
    "C23_trihydroxy": 395.2792,
    "C23_tetrahydroxy": 411.2741,
    "C23_pentahydroxy": 427.269,

    "C27_monohydroxy": 419.352,
    "C27_dihydroxy": 435.3469,
    "C27_trihydroxy": 451.3418,
    "C27_tetrahydroxy": 467.3367,
    "C27_pentahydroxy": 483.3316,

    "Aceto_dihydroxy": 435.3105,

    "C24_alkylamine_monohydroxy": 377.305,
    "C24_alkylamine_dihydroxy": 393.2999,
    "C24_alkylamine_trihydroxy_combined": 409.2949,
    "C24_alkylamine_tetrahydroxy": 425.2898,
    "C24_alkylamine_pentahydroxy": 441.2847,
}

# it has to be at least two scans not from synthesis project.
synthesis = [
    "MSV000099509",
    "MSV000100947",
    "MSV000100886",
    "MSV000100534",
    "MSV000100532",
    "MSV000100441",
    "MSV000100347",
    "MSV000100095",
    "MSV000099376",
    "MSV000099375",
    "MSV000099242",
    "MSV000099090",
    "MSV000098640",
    "MSV000098639",
    "MSV000098637",
    "MSV000098628",
    "MSV000097885",
    "MSV000097869",
    "MSV000095820",
    "MSV000094447",
    "MSV000094393",
    "MSV000094391",
]

In [ ]:
entropy_files = {
    "C24_nonhydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C24_nonhydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C24_monohydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C24_monohydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C24_dihydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C24_dihydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C24_trihydroxy_combined": f"{PROJECT_DIR}/Entropy/entropy_results/C24_final_trihydroxy_combined_renumbered_spectral_entropy.tsv",
    "C24_tetrahydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C24_tetrahydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C24_pentahydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C24_pentahydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",

    "C23_monohydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C23_monohydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C23_dihydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C23_dihydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C23_trihydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C23_trihydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C23_tetrahydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C23_tetrahydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C23_pentahydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C23_pentahydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",

    "C27_monohydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C27_monohydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C27_dihydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C27_dihydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C27_trihydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C27_trihydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C27_tetrahydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C27_tetrahydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C27_pentahydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C27_pentahydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",

    "Aceto_dihydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/aceto_dihydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",

    "C24_alkylamine_monohydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C24_alkylamine_monohydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C24_alkylamine_dihydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C24_alkylamine_dihydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C24_alkylamine_trihydroxy_combined": f"{PROJECT_DIR}/Entropy/entropy_results/C24_alkylamine_trihydroxy_falcon_eps01_minsamples2_combined_renumbered_spectral_entropy.tsv",
    "C24_alkylamine_tetrahydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C24_alkylamine_tetrahydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C24_alkylamine_pentahydroxy": f"{PROJECT_DIR}/Entropy/entropy_results/C24_alkylamine_pentahydroxy_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",

    "C28_alcohol_1OH": f"{PROJECT_DIR}/Entropy/entropy_results/C28_alcohol_1OH_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C28_alcohol_2OH": f"{PROJECT_DIR}/Entropy/entropy_results/C28_alcohol_2OH_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
    "C28_alcohol_3OH": f"{PROJECT_DIR}/Entropy/entropy_results/C28_alcohol_3OH_falcon_eps01_minsamples2_scan_spectral_entropy.tsv",
}

In [ ]:
# Entropy-filtered post-clustering summary for all tables
import os
import pandas as pd

base_dir = PROJECT_DIR
synthesis_set = set(synthesis)


def resolve_path(path):
    return path if os.path.isabs(path) else os.path.join(base_dir, path)


def keep_cluster(group):
    in_list = group['dataset'].isin(synthesis_set)
    not_in_list = ~in_list

    # If all datasets are from the synthesis list, remove.
    if in_list.all():
        return False

    # If mixed, keep only if at least 2 rows are not in the list.
    if not_in_list.sum() >= 2:
        return True

    return False


def format_remaining_scans(scan_series):
    scan_values = scan_series.dropna()

    # Prefer numeric ordering when scan IDs are numeric.
    numeric_values = pd.to_numeric(scan_values, errors='coerce')
    if numeric_values.notna().all():
        ordered = sorted(numeric_values.astype(int).astype(str).unique(), key=int)
        return ';'.join(ordered)

    string_values = scan_values.astype(str).unique().tolist()
    string_values = sorted(string_values)
    return ';'.join(string_values)


combined_configs = {
    'C24_trihydroxy_combined': {
        'part_csv_keys': ['C24_trihydroxy_part1', 'C24_trihydroxy_part2', 'C24_trihydroxy_part3'],
        'part_annotation_files': [
            f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Trihydroxy/download_mgfs/final_mgf/C24_Trihydroxy_input_falcon_charge_part1_annotation_table.tsv',
            f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Trihydroxy/download_mgfs/final_mgf/C24_Trihydroxy_input_falcon_charge_part2_annotation_table.tsv',
            f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Trihydroxy/download_mgfs/final_mgf/C24_Trihydroxy_input_falcon_charge_part3_annotation_table.tsv',
        ],
        'scan_mgf_names': [
            'C24_final_trihydroxy_part1_falcon_eps01_minsamples2_scan.mgf',
            'C24_final_trihydroxy_part2_falcon_eps01_minsamples2_scan.mgf',
            'C24_final_trihydroxy_part3_falcon_eps01_minsamples2_scan.mgf',
        ],
        'scan_mapping_file': f'{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_final_trihydroxy_combined_scan_mapping.tsv',
    },
    'C24_alkylamine_trihydroxy_combined': {
        'part_csv_keys': [
            'C24_alkylamine_trihydroxy_part1',
            'C24_alkylamine_trihydroxy_part2',
            'C24_alkylamine_trihydroxy_part3',
            'C24_alkylamine_trihydroxy_part4',
        ],
        'part_annotation_files': [
            f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part1_annotation_table.tsv',
            f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part2_annotation_table.tsv',
            f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part3_annotation_table.tsv',
            f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part4_annotation_table.tsv',
        ],
        'scan_mgf_names': [
            'C24_alkylamine_trihydroxy_part1_falcon_eps01_minsamples2_scan.mgf',
            'C24_alkylamine_trihydroxy_part2_falcon_eps01_minsamples2_scan.mgf',
            'C24_alkylamine_trihydroxy_part3_falcon_eps01_minsamples2_scan.mgf',
            'C24_alkylamine_trihydroxy_part4_falcon_eps01_minsamples2_scan.mgf',
        ],
        'scan_mapping_file': f'{PROJECT_DIR}/Falcon/all_falcon_outputs/C24_alkylamine_trihydroxy_combined_scan_mapping.tsv',
    },
}

# Fallback values in case the active deltas_calculation dictionary uses part keys only.
combined_delta_overrides = {
    'C24_trihydroxy_combined': 409.2949,
    'C24_alkylamine_trihydroxy_combined': 409.2949,
}

summary_rows_entropy = []

for table_name, entropy_path in entropy_files.items():
    entropy = pd.read_csv(resolve_path(entropy_path), sep='\t')
    entropy_threshold = 7.0 if table_name == 'Aceto_dihydroxy' else 7.5

    if table_name in combined_configs:
        config = combined_configs[table_name]
        combined_frames = []

        for part_csv_key, part_annotation_path, scan_mgf_name in zip(
            config['part_csv_keys'],
            config['part_annotation_files'],
            config['scan_mgf_names'],
        ):
            df = pd.read_csv(resolve_path(falcon_csv_files[part_csv_key]), sep=',', skiprows=25)
            df['cluster'] = df['cluster'] + 1
            df = df[df['cluster'] > 0]
            df['scan_combined_mgf'] = df['identifier'].str.split(':').str[-1].astype(int)
            df['cluster_label'] = part_csv_key + ':' + df['cluster'].astype(int).astype(str)
            df['filename_scan'] = scan_mgf_name + ':' + df['cluster'].astype(int).astype(str)

            annotation_table = pd.read_csv(resolve_path(part_annotation_path), sep='\t')
            annotation_table.columns = annotation_table.columns.str.strip()

            part_merged = pd.merge(df, annotation_table, on='scan_combined_mgf', how='left')
            part_merged['dataset'] = part_merged['original_path'].str.split('/').str[2]
            combined_frames.append(part_merged)

        merged_df = pd.concat(combined_frames, ignore_index=True)
        merged_df = merged_df.groupby('cluster_label').filter(keep_cluster)

        mapping_df = pd.read_csv(resolve_path(config['scan_mapping_file']), sep='\t')
        entropy = entropy.rename(columns={'scan_number': 'scan_entropy'})

        merged_entropy = pd.merge(merged_df, mapping_df, on='filename_scan', how='left')
        merged_entropy = pd.merge(merged_entropy, entropy, on='scan_entropy', how='left')
        merged_entropy_filtered = merged_entropy[~(merged_entropy['entropy_score'] > entropy_threshold)].copy()

        n_scans_entropy = merged_entropy_filtered['cluster_label'].nunique()
        # For combined groups, report the mapped combined scan IDs from the scan_mapping TSV files.
        remaining_scans = format_remaining_scans(merged_entropy_filtered['scan_entropy'])

    else:
        if table_name not in falcon_csv_files or table_name not in annotation_table_pre_falcon:
            print(f"{table_name}: skipped (missing table in falcon_csv_files or annotation_table_pre_falcon)")
            print('-' * 100)
            summary_rows_entropy.append({
                'table_name': table_name,
                'n_scans_post_cluster_entropy_filter': 'skipped',
                'n_deltas_post_cluster_entropy_filter': 'skipped',
                'entropy_threshold': entropy_threshold,
                'remaining_scans_after_filters': 'skipped',
            })
            continue

        df = pd.read_csv(resolve_path(falcon_csv_files[table_name]), sep=',', skiprows=25)
        df['cluster'] = df['cluster'] + 1
        df = df[df['cluster'] > 0]
        df['scan_combined_mgf'] = df['identifier'].str.split(':').str[-1].astype(int)
        df['cluster_label'] = df['cluster'].astype(int).astype(str)

        annotation_table = pd.read_csv(resolve_path(annotation_table_pre_falcon[table_name]), sep='\t')
        annotation_table.columns = annotation_table.columns.str.strip()

        merged_df = pd.merge(df, annotation_table, on='scan_combined_mgf', how='left')
        merged_df['dataset'] = merged_df['original_path'].str.split('/').str[2]
        merged_df = merged_df.groupby('cluster_label').filter(keep_cluster)

        entropy = entropy.rename(columns={'scan_number': 'cluster'})
        merged_entropy = pd.merge(merged_df, entropy, on='cluster', how='left')
        merged_entropy_filtered = merged_entropy[~(merged_entropy['entropy_score'] > entropy_threshold)].copy()

        n_scans_entropy = merged_entropy_filtered['cluster_label'].nunique()
        remaining_scans = format_remaining_scans(merged_entropy_filtered['cluster_label'])

    print(f"{table_name}: number of scans after synthesis + entropy filter, post-cluster: {n_scans_entropy}")

    delta_to_subtract = deltas_calculation.get(table_name)
    if delta_to_subtract is None:
        delta_to_subtract = combined_delta_overrides.get(table_name)

    if delta_to_subtract is None or pd.isna(delta_to_subtract):
        n_deltas_entropy = 'unspecified'
    else:
        merged_entropy_filtered['delta'] = merged_entropy_filtered['pepmass'] - delta_to_subtract
        merged_entropy_filtered['delta_rounded'] = merged_entropy_filtered['delta'].round(2)
        n_deltas_entropy = merged_entropy_filtered['delta_rounded'].nunique()

    print(f"{table_name}: number of deltas after synthesis + entropy filter, post-cluster: {n_deltas_entropy}")
    # print(f"{table_name}: remaining scans after filters: {remaining_scans}")
    print('-' * 100)

    summary_rows_entropy.append({
        'table_name': table_name,
        'n_scans_post_cluster_entropy_filter': n_scans_entropy,
        'n_deltas_post_cluster_entropy_filter': n_deltas_entropy,
        'entropy_threshold': entropy_threshold,
        'remaining_scans_after_filters': remaining_scans,
    })

summary_entropy_df = pd.DataFrame(summary_rows_entropy)
summary_entropy_outfile = f'{PROJECT_DIR}/Scripts_data_analysis/deltas_calculation/deltas_postclustering_entropy_summary.tsv'
remaining_scans_outfile = f'{PROJECT_DIR}/Scripts_data_analysis/deltas_calculation/deltas_postclustering_entropy_remaining_scans.tsv'

# Main summary without the remaining scan IDs.
summary_entropy_export_df = summary_entropy_df.drop(columns=['remaining_scans_after_filters'])
summary_entropy_export_df.to_csv(summary_entropy_outfile, sep='\t', index=False)

# Separate table with only remaining scans per table.
remaining_scans_df = summary_entropy_df[['table_name', 'remaining_scans_after_filters']].copy()
remaining_scans_df.to_csv(remaining_scans_outfile, sep='\t', index=False)

print(f"Entropy summary table exported to: {summary_entropy_outfile}")
print(f"Remaining scans table exported to: {remaining_scans_outfile}")
summary_entropy_df

# Molecular Networking filter - WITH SPECREBOOT

In [ ]:
graphml_network = {
    'c24_nonhydroxy': f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/c24_nonhydroxy_gnps_ff_gnps_threshold.graphml', 
    'c24_monohydroxy': f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/c24_monohydroxy_gnps_ff_gnps_threshold.graphml',
    'c24_dihydroxy': f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/c24_dihydroxy_gnps_ff_gnps_threshold.graphml',
    'c24_trihydroxy': f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/c24_trihydroxy_combined_gnps_threshold.graphml',
    'c24_tetrahydroxy': f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/c24_tetrahydroxy_gnps_ff_gnps_threshold.graphml',
    'c24_pentahydroxy': f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/c24_pentahydroxy_gnps_ff_gnps_threshold.graphml',

    "C23_monohydroxy": f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C23_monohydroxy_gnps_ff_gnps_threshold.graphml',
    "C23_dihydroxy": f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C23_dihydroxy_gnps_ff_gnps_threshold.graphml',
    "C23_trihydroxy": f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C23_trihydroxy_gnps_ff_gnps_threshold.graphml',
    "C23_tetrahydroxy": f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C23_tetrahydroxy_gnps_ff_gnps_threshold.graphml',
    "C23_pentahydroxy": f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C23_pentahydroxy_gnps_ff_gnps_threshold.graphml',

    "C27_monohydroxy": f"{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C27_monohydroxy_gnps_ff_gnps_threshold.graphml",
    "C27_dihydroxy": f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C27_dihydroxy_gnps_ff_gnps_threshold.graphml',
    "C27_trihydroxy": f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C27_trihydroxy_gnps_ff_gnps_threshold.graphml',
    "C27_tetrahydroxy": f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C27_tetrahydroxy_gnps_ff_gnps_threshold.graphml',
    "C27_pentahydroxy": f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C27_pentahydroxy_gnps_ff_gnps_threshold.graphml',

    "Aceto_dihydroxy": f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/aceto_dihydroxy_gnps_ff_gnps_threshold.graphml',

    "C24_alkylamine_monohydroxy": f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C24_alkylamine_monohydroxy_gnps_ff_gnps_threshold.graphml',
    "C24_alkylamine_dihydroxy": f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C24_alkylamine_dihydroxy_gnps_ff_gnps_threshold.graphml',
    "C24_alkylamine_trihydroxy_combined": f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C24_alkylamine_trihydroxy_combined_gnps_ff_gnps_threshold.graphml',
    "C24_alkylamine_tetrahydroxy": f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C24_alkylamine_tetrahydroxy_gnps_ff_gnps_threshold.graphml',
    "C24_alkylamine_pentahydroxy": f"{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C24_alkylamine_pentahydroxy_gnps_ff_gnps_threshold.graphml",

    "C28_alcohol_2OH": f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C28_alcohol_2OH_gnps_ff_gnps_threshold.graphml',
    "C28_alcohol_3OH": f'{PROJECT_DIR}/postMN_final_filter/SpecReBoot/graphml_files/C28_alcohol_3OH_gnps_ff_gnps_threshold.graphml',
}

annotation_files = {
    'c24_nonhydroxy': f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C24_-_Nonhydroxy.tsv',
    'c24_monohydroxy': f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C24_-_Monohydroxy.tsv',
    'c24_dihydroxy': f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C24_-_Dihydroxy.tsv',
    'c24_trihydroxy': f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C24_-_Trihydroxy_combined.tsv',
    'c24_tetrahydroxy': f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C24_-_Tetrahydroxy.tsv',
    'c24_pentahydroxy': f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C24_-_Pentahydroxy.tsv',

    "C23_monohydroxy": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C23_-_Monohydroxy.tsv',
    "C23_dihydroxy": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C23_-_Dihydroxy.tsv',
    "C23_trihydroxy": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C23_-_Trihydroxy.tsv',
    "C23_tetrahydroxy": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C23_-_Tetrahydroxy.tsv',
    "C23_pentahydroxy": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C23_-_Pentahydroxy.tsv',

    "C27_monohydroxy": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C27_-_Monohydroxy.tsv',
    "C27_dihydroxy": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C27_-_Dihydroxy.tsv',
    "C27_trihydroxy": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C27_-_Trihydroxy.tsv',
    "C27_tetrahydroxy": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C27_-_Tetrahydroxy.tsv',
    "C27_pentahydroxy": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C27_-_Pentahydroxy.tsv',

    "Aceto_dihydroxy": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/Aceto_dihydroxy.tsv',

    "C24_alkylamine_monohydroxy": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C24_alkylamine_-_Monohydroxy.tsv',
    "C24_alkylamine_dihydroxy": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C24_alkylamine_-_Dihydroxy.tsv',
    "C24_alkylamine_trihydroxy_combined": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C24_alkylamine_-_Trihydroxy_combined.tsv',
    "C24_alkylamine_tetrahydroxy": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C24_alkamine_-_Tetrahydroxy.tsv',
    "C24_alkylamine_pentahydroxy": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C24_alkylamine_-_Pentahydroxy.tsv',

    "C28_alcohol_2OH": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C28_alcohol_-_2OH.tsv',
    "C28_alcohol_3OH": f'{PROJECT_DIR}/library_results_postMN_filter/annotations_MN_per_core/C28_alcohol_-_3OH.tsv',
}

mgf_files_input = {
    'c24_nonhydroxy': f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_nonhydroxy_falcon_eps01_minsamples2_scan.mgf',
    'c24_monohydroxy': f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_monohydroxy_falcon_eps01_minsamples2_scan.mgf',
    'c24_dihydroxy': f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_dihydroxy_falcon_eps01_minsamples2_scan.mgf',
    'c24_trihydroxy': f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_final_trihydroxy_combined_renumbered.mgf',
    'c24_tetrahydroxy': f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_tetrahydroxy_falcon_eps01_minsamples2_scan.mgf',
    'c24_pentahydroxy': f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_pentahydroxy_falcon_eps01_minsamples2_scan.mgf',

    "C23_monohydroxy": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C23_monohydroxy_falcon_eps01_minsamples2_scan.mgf',
    "C23_dihydroxy": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C23_dihydroxy_falcon_eps01_minsamples2_scan.mgf',
    "C23_trihydroxy": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C23_trihydroxy_falcon_eps01_minsamples2_scan.mgf',
    "C23_tetrahydroxy": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C23_tetrahydroxy_falcon_eps01_minsamples2_scan.mgf',
    "C23_pentahydroxy": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C23_pentahydroxy_falcon_eps01_minsamples2_scan.mgf',

    "C27_monohydroxy": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C27_monohydroxy_falcon_eps01_minsamples2_scan.mgf',
    "C27_dihydroxy": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C27_dihydroxy_falcon_eps01_minsamples2_scan.mgf',
    "C27_trihydroxy": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C27_trihydroxy_falcon_eps01_minsamples2_scan.mgf',
    "C27_tetrahydroxy": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C27_tetrahydroxy_falcon_eps01_minsamples2_scan.mgf',
    "C27_pentahydroxy": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C27_pentahydroxy_falcon_eps01_minsamples2_scan.mgf',

    "Aceto_dihydroxy": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/aceto_dihydroxy_falcon_eps01_minsamples2_scan.mgf',

    "C24_alkylamine_monohydroxy": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_alkylamine_monohydroxy_falcon_eps01_minsamples2_scan.mgf',
    "C24_alkylamine_dihydroxy": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_alkylamine_dihydroxy_falcon_eps01_minsamples2_scan.mgf',
    "C24_alkylamine_trihydroxy_combined": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_alkylamine_trihydroxy_falcon_eps01_minsamples2_combined_renumbered.mgf',
    "C24_alkylamine_tetrahydroxy": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_alkylamine_tetrahydroxy_falcon_eps01_minsamples2_scan.mgf',
    "C24_alkylamine_pentahydroxy": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C24_alkylamine_pentahydroxy_falcon_eps01_minsamples2_scan.mgf',

    "C28_alcohol_2OH": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C28_alcohol_2OH_falcon_eps01_minsamples2_scan.mgf',
    "C28_alcohol_3OH": f'{PROJECT_DIR}/Falcon/all_falcon_outputs/consider_these/C28_alcohol_3OH_falcon_eps01_minsamples2_scan.mgf',
}

mgf_files_output = {
    'c24_nonhydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_nonhydroxy_filtered_MN.mgf',
    'c24_monohydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_monohydroxy_filtered_MN.mgf',
    'c24_dihydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_dihydroxy_filtered_MN.mgf',
    'c24_trihydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_trihydroxy_combined_filtered_MN.mgf',
    'c24_tetrahydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_tetrahydroxy_filtered_MN.mgf',
    'c24_pentahydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_pentahydroxy_filtered_MN.mgf',

    "C23_monohydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C23_monohydroxy_filtered_MN.mgf',
    "C23_dihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C23_dihydroxy_filtered_MN.mgf',
    "C23_trihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C23_trihydroxy_filtered_MN.mgf',
    "C23_tetrahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C23_tetrahydroxy_filtered_MN.mgf',
    "C23_pentahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C23_pentahydroxy_filtered_MN.mgf',

    "C27_monohydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C27_monohydroxy_filtered_MN.mgf',
    "C27_dihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C27_dihydroxy_filtered_MN.mgf',
    "C27_trihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C27_trihydroxy_filtered_MN.mgf',
    "C27_tetrahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C27_tetrahydroxy_filtered_MN.mgf',
    "C27_pentahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C27_pentahydroxy_filtered_MN.mgf',

    "Aceto_dihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/Aceto_dihydroxy_filtered_MN.mgf',

    "C24_alkylamine_monohydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_alkylamine_monohydroxy_filtered_MN.mgf',
    "C24_alkylamine_dihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_alkylamine_dihydroxy_filtered_MN.mgf',
    "C24_alkylamine_trihydroxy_combined": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_alkylamine_trihydroxy_combined_filtered_MN.mgf',
    "C24_alkylamine_tetrahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_alkylamine_tetrahydroxy_filtered_MN.mgf',
    "C24_alkylamine_pentahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_alkylamine_pentahydroxy_filtered_MN.mgf',

    "C28_alcohol_2OH": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C28_alcohol_2OH_filtered_MN.mgf',
    "C28_alcohol_3OH": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C28_alcohol_3OH_filtered_MN.mgf',
}

In [ ]:
from pathlib import Path
import csv
from collections import defaultdict
import xml.etree.ElementTree as ET
import os
import pandas as pd

def extract_node_component_table(graphml_path, output_tsv_path=None):
    """Extract node-component information and label degree-0 nodes as component -1."""
    graphml_path = Path(graphml_path).expanduser()
    if not graphml_path.exists():
        raise FileNotFoundError(f"GraphML file not found: {graphml_path}")

    if output_tsv_path is None:
        output_tsv_path = graphml_path.with_name(graphml_path.stem + "_node_component.tsv")
    else:
        output_tsv_path = Path(output_tsv_path).expanduser()

    # Ensure output folder exists for per-group generated tables.
    output_tsv_path.parent.mkdir(parents=True, exist_ok=True)

    ns = "{http://graphml.graphdrawing.org/xmlns}"

    # Find which key ID stores the node-level component attribute.
    component_key = None
    for _, elem in ET.iterparse(graphml_path, events=("start",)):
        if (
            elem.tag == f"{ns}key"
            and elem.get("for") == "node"
            and elem.get("attr.name") == "component"
        ):
            component_key = elem.get("id")
            break

    if component_key is None:
        raise ValueError("No node-level 'component' attribute found in this GraphML file.")

    node_components = {}
    node_order = []
    degree = defaultdict(int)

    # Parse nodes and edges in one streaming pass.
    for _, elem in ET.iterparse(graphml_path, events=("end",)):
        if elem.tag == f"{ns}node":
            node_id = elem.get("id")
            graphml_component = ""
            for data in elem.findall(f"{ns}data"):
                if data.get("key") == component_key:
                    graphml_component = (data.text or "").strip()
                    break
            node_components[node_id] = graphml_component
            node_order.append(node_id)
            elem.clear()
        elif elem.tag == f"{ns}edge":
            src = elem.get("source")
            tgt = elem.get("target")
            if src is not None:
                degree[src] += 1
            if tgt is not None:
                degree[tgt] += 1
            elem.clear()

    singleton_count = 0
    graphml_components_unique = set()
    output_components_unique = set()

    with output_tsv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle, delimiter="\t")
        writer.writerow(["node_id", "component", "graphml_component", "degree", "is_singleton"])

        for node_id in node_order:
            graphml_component = node_components.get(node_id, "")
            node_degree = degree.get(node_id, 0)
            is_singleton = 1 if node_degree == 0 else 0
            output_component = "-1" if is_singleton else graphml_component

            if is_singleton:
                singleton_count += 1

            graphml_components_unique.add(graphml_component)
            output_components_unique.add(output_component)

            writer.writerow([node_id, output_component, graphml_component, node_degree, is_singleton])

    node_count = len(node_order)
    return (
        output_tsv_path,
        node_count,
        singleton_count,
        len(graphml_components_unique),
        len(output_components_unique),
    )

In [ ]:
def filter_mgf_from_annotations(cluster_summary_path, annotation_path, mgf_input_path, mgf_output_path, group_name):
    # Load and harmonize input tables (works for classic cluster summary and GraphML-derived node_component TSV).
    df = pd.read_csv(cluster_summary_path, sep='\t')
    annotation = pd.read_csv(annotation_path, sep='\t')

    # Standardize scan column name.
    if 'cluster index' in df.columns:
        df = df.rename(columns={'cluster index': 'Scan'})
    elif 'node_id' in df.columns:
        df = df.rename(columns={'node_id': 'Scan'})
    elif 'Scan' not in df.columns:
        raise ValueError(f"No scan column found in {cluster_summary_path}. Expected one of: 'cluster index', 'node_id', 'Scan'.")

    if '#Scan#' in annotation.columns:
        annotation = annotation.rename(columns={'#Scan#': 'Scan'})
    elif 'Scan' not in annotation.columns:
        raise ValueError(f"No scan column found in {annotation_path}. Expected '#Scan#' or 'Scan'.")

    # Keep only required columns and make types comparable.
    if 'component' not in df.columns:
        raise ValueError(f"Column 'component' not found in {cluster_summary_path}.")
    df = df[['Scan', 'component']].copy()
    df['Scan'] = df['Scan'].astype(str)
    df['component'] = df['component'].astype(str)

    annotation = annotation[['Scan', 'Is_BA_Checked']].copy()
    annotation['Scan'] = annotation['Scan'].astype(str)
    annotation['Is_BA_Checked'] = annotation['Is_BA_Checked'].astype(str).str.strip().str.lower()

    merged = pd.merge(df, annotation, on='Scan', how='left')

    # ------------------------------------------------------------------
    # 1/3 fractional removal logic (ported from the fractional script).
    # NOTE: 'component' is a string here, so singletons are '-1' (string).
    # The yes/no fraction uses ONLY rows labeled 'yes' or 'no' as the
    # denominator; blanks / 'nan' / unlabeled rows are ignored in the ratio.
    # ------------------------------------------------------------------

    # Compute yes/no proportions per component using only rows labeled yes/no.
    component_counts = (
        merged[merged['Is_BA_Checked'].isin(['yes', 'no'])]
        .groupby('component')['Is_BA_Checked']
        .value_counts()
        .unstack(fill_value=0)
        .reindex(columns=['yes', 'no'], fill_value=0)
    )
    component_counts['total_yes_no'] = component_counts['yes'] + component_counts['no']
    component_counts['no_fraction'] = component_counts['no'] / component_counts['total_yes_no']

    # Components that contain at least one 'no' (excluding singletons).
    components_with_no = set(
        component_counts[(component_counts['no'] > 0) & (component_counts.index != '-1')].index
    )

    # If >33% of the classified ones are 'no', remove ALL nodes in that component.
    components_remove_all = set(
        component_counts[
            (component_counts['no'] > 0)
            & (component_counts['no_fraction'] > 0.33)
            & (component_counts.index != '-1')
        ].index
    )

    # If <=33% are 'no', remove ONLY the nodes marked 'no'.
    components_remove_only_no = components_with_no - components_remove_all

    remove_mask = (
        merged['component'].isin(components_remove_all)
        | (
            merged['component'].isin(components_remove_only_no)
            & (merged['Is_BA_Checked'] == 'no')
        )
    ) | ((merged['component'] == '-1') & (merged['Is_BA_Checked'] == 'no'))

    scans_to_remove = merged.loc[remove_mask, 'Scan'].dropna().unique().tolist()
    features_to_keep = df[~df['Scan'].isin(scans_to_remove)]['Scan'].unique().tolist()

    # Save features-to-keep list next to the output MGF.
    output_dir = os.path.dirname(mgf_output_path)
    os.makedirs(output_dir, exist_ok=True)
    features_output_path = os.path.join(output_dir, f'{group_name}_features_to_keep.txt')
    pd.Series(features_to_keep).to_csv(features_output_path, sep='\t', index=False, header=False)

    ids_to_keep = set(str(scan_id) for scan_id in features_to_keep)

    # Filter MGF by keeping full BEGIN/END IONS blocks matching kept SCANS IDs.
    kept_spectra = 0
    total_spectra = 0
    with open(mgf_input_path, 'r') as fin, open(mgf_output_path, 'w') as fout:
        current_block = []
        current_scan = None
        in_block = False

        for line in fin:
            if line.startswith('BEGIN IONS'):
                in_block = True
                current_block = [line]
                current_scan = None
                continue

            if in_block:
                current_block.append(line)
                if line.startswith('SCANS='):
                    current_scan = line.strip().split('=', 1)[1]
                if line.startswith('END IONS'):
                    total_spectra += 1
                    if current_scan in ids_to_keep:
                        fout.writelines(current_block)
                        kept_spectra += 1
                    in_block = False
                    current_block = []
            else:
                # Preserve any non-spectrum lines outside BEGIN/END IONS blocks.
                fout.write(line)

    return {
        'group': group_name,
        'features_to_keep': len(features_to_keep),
        'input_spectra': total_spectra,
        'kept_spectra': kept_spectra,
        'features_file': features_output_path,
        'mgf_output': mgf_output_path,
    }


required_dicts = {
    'graphml_network': graphml_network,
    'annotation_files': annotation_files,
    'mgf_files_input': mgf_files_input,
    'mgf_files_output': mgf_files_output,
}

common_groups = set.intersection(*(set(d.keys()) for d in required_dicts.values()))
if not common_groups:
    raise ValueError('No overlapping group names were found across the filepath dictionaries.')

missing_by_dict = {
    dict_name: sorted(set(common_groups) ^ set(path_dict.keys()))
    for dict_name, path_dict in required_dicts.items()
}
for dict_name, missing in missing_by_dict.items():
    if missing:
        warnings.warn(f'{dict_name} has key mismatches relative to the common groups: {missing}')

results = []
skipped_groups = []
generated_component_tables = {}

for group in sorted(common_groups):
    paths = {
        'graphml': graphml_network[group],
        'annotation': annotation_files[group],
        'mgf_input': mgf_files_input[group],
    }
    missing_files = [label for label, path in paths.items() if not os.path.exists(path)]
    if missing_files:
        warnings.warn(f"Skipping {group}: missing files for {', '.join(missing_files)}")
        skipped_groups.append(group)
        continue

    # Build a per-group node_component table from GraphML, then use it as the cluster table input.
    component_table_path = os.path.join(
        os.path.dirname(mgf_files_output[group]),
        f'{group}_node_component.tsv',
    )
    (
        output_path,
        node_count,
        singleton_count,
        graphml_component_count,
        output_component_count,
    ) = extract_node_component_table(
        graphml_path=graphml_network[group],
        output_tsv_path=component_table_path,
    )

    generated_component_tables[group] = str(output_path)

    result = filter_mgf_from_annotations(
        cluster_summary_path=str(output_path),
        annotation_path=annotation_files[group],
        mgf_input_path=mgf_files_input[group],
        mgf_output_path=mgf_files_output[group],
        group_name=group,
    )
    result.update({
        'component_table': str(output_path),
        'node_count': node_count,
        'singleton_count': singleton_count,
        'graphml_component_count': graphml_component_count,
        'output_component_count': output_component_count,
    })
    results.append(result)

results_df = pd.DataFrame(results)
print(f'Processed groups: {len(results)}')
print(f'Skipped groups: {len(skipped_groups)}')
if skipped_groups:
    print('Skipped:', ', '.join(skipped_groups))

results_df
results_df.to_csv(f'{PROJECT_DIR}/Scripts_data_analysis/molecular_networking_filter/All_with_SpecReBoot.tsv', sep='\t', index=False)

In [ ]:
#add the information of which scans for each core are kept

from pathlib import Path

features_to_keep_dict = {
    'c24_nonhydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/c24_nonhydroxy_features_to_keep.txt',
    'c24_monohydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/c24_monohydroxy_features_to_keep.txt',
    'c24_dihydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/c24_dihydroxy_features_to_keep.txt',
    'c24_trihydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/c24_trihydroxy_features_to_keep.txt',
    'c24_tetrahydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/c24_tetrahydroxy_features_to_keep.txt',
    'c24_pentahydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/c24_pentahydroxy_features_to_keep.txt',

    'C23_monohydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C23_monohydroxy_features_to_keep.txt',
    'C23_dihydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C23_dihydroxy_features_to_keep.txt',
    'C23_trihydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C23_trihydroxy_features_to_keep.txt',
    'C23_tetrahydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C23_tetrahydroxy_features_to_keep.txt',
    'C23_pentahydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C23_pentahydroxy_features_to_keep.txt',

    'C27_monohydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C27_monohydroxy_features_to_keep.txt',
    'C27_dihydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C27_dihydroxy_features_to_keep.txt',
    'C27_trihydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C27_trihydroxy_features_to_keep.txt',
    'C27_tetrahydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C27_tetrahydroxy_features_to_keep.txt',
    'C27_pentahydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C27_pentahydroxy_features_to_keep.txt',

    'Aceto_dihydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/Aceto_dihydroxy_features_to_keep.txt',

    'C24_alkylamine_monohydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_alkylamine_monohydroxy_features_to_keep.txt',
    'C24_alkylamine_dihydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_alkylamine_dihydroxy_features_to_keep.txt',
    'C24_alkylamine_trihydroxy_combined': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_alkylamine_trihydroxy_combined_features_to_keep.txt',
    'C24_alkylamine_tetrahydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_alkylamine_tetrahydroxy_features_to_keep.txt',
    'C24_alkylamine_pentahydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_alkylamine_pentahydroxy_features_to_keep.txt',

    'C28_alcohol_2OH': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C28_alcohol_2OH_features_to_keep.txt',
    'C28_alcohol_3OH': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C28_alcohol_3OH_features_to_keep.txt',
}

df = pd.read_csv(
    f'{PROJECT_DIR}/Scripts_data_analysis/molecular_networking_filter/All_with_SpecReBoot.tsv',
    sep='\t'
)

remaining_scans_by_group = {}

for group, features_file in features_to_keep_dict.items():
    file_path = Path(features_file)
    if not file_path.exists():
        remaining_scans_by_group[group] = ''
        continue

    scans = pd.read_csv(file_path, sep='\t', header=None).iloc[:, 0]
    scan_list = scans.dropna().astype(str).tolist()
    remaining_scans_by_group[group] = ';'.join(scan_list)

df['remaining_scans_after_MN_filter'] = df['group'].map(remaining_scans_by_group).fillna('')

df.to_csv(
    f'{PROJECT_DIR}/Scripts_data_analysis/molecular_networking_filter/All_with_SpecReBoot_and_remaining_scans.tsv',
    sep='\t',
    index=False
)

df

### Now let's see which survive BOTH the molecular networking AND the entropy filter (previous step)

In [ ]:
import pandas as pd

mgf_post_MN_filter = {
    'c24_nonhydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_nonhydroxy_filtered_MN.mgf',
    'c24_monohydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_monohydroxy_filtered_MN.mgf',
    'c24_dihydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_dihydroxy_filtered_MN.mgf',
    'c24_trihydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_trihydroxy_combined_filtered_MN.mgf',
    'c24_tetrahydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_tetrahydroxy_filtered_MN.mgf',
    'c24_pentahydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_pentahydroxy_filtered_MN.mgf',

    "C23_monohydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C23_monohydroxy_filtered_MN.mgf',
    "C23_dihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C23_dihydroxy_filtered_MN.mgf',
    "C23_trihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C23_trihydroxy_filtered_MN.mgf',
    "C23_tetrahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C23_tetrahydroxy_filtered_MN.mgf',
    "C23_pentahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C23_pentahydroxy_filtered_MN.mgf',

    "C27_monohydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C27_monohydroxy_filtered_MN.mgf',
    "C27_dihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C27_dihydroxy_filtered_MN.mgf',
    "C27_trihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C27_trihydroxy_filtered_MN.mgf',
    "C27_tetrahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C27_tetrahydroxy_filtered_MN.mgf',
    "C27_pentahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C27_pentahydroxy_filtered_MN.mgf',

    "Aceto_dihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/Aceto_dihydroxy_filtered_MN.mgf',

    "C24_alkylamine_monohydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_alkylamine_monohydroxy_filtered_MN.mgf',
    "C24_alkylamine_dihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_alkylamine_dihydroxy_filtered_MN.mgf',
    "C24_alkylamine_trihydroxy_combined": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_alkylamine_trihydroxy_combined_filtered_MN.mgf',
    "C24_alkylamine_tetrahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_alkylamine_tetrahydroxy_filtered_MN.mgf',
    "C24_alkylamine_pentahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C24_alkylamine_pentahydroxy_filtered_MN.mgf',

    # "C28_alcohol_1OH": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C28_alcohol_1OH_falcon_eps01_minsamples2_scan.mgf',
    "C28_alcohol_2OH": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C28_alcohol_2OH_filtered_MN.mgf',
    "C28_alcohol_3OH": f'{PROJECT_DIR}/Falcon/postMN_final_filter/SpecReBoot/mgf_filtered_MN/C28_alcohol_3OH_filtered_MN.mgf',
}

mgf_post_entropy_and_MN = {
    'c24_nonhydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_nonhydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    'c24_monohydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_monohydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    'c24_dihydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_dihydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    'c24_trihydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_trihydroxy_combined_SpecReBoot_filtered_MN_and_entropy.mgf',
    'c24_tetrahydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_tetrahydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    'c24_pentahydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_pentahydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',

    "C23_monohydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C23_monohydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    "C23_dihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C23_dihydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    "C23_trihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C23_trihydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    "C23_tetrahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C23_tetrahydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    "C23_pentahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C23_pentahydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',

    "C27_monohydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C27_monohydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    "C27_dihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C27_dihydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    "C27_trihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C27_trihydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    "C27_tetrahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C27_tetrahydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    "C27_pentahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C27_pentahydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',

    "Aceto_dihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/Aceto_dihydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',

    "C24_alkylamine_monohydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_alkylamine_monohydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    "C24_alkylamine_dihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_alkylamine_dihydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    "C24_alkylamine_trihydroxy_combined": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_alkylamine_trihydroxy_combined_SpecReBoot_filtered_MN_and_entropy.mgf',
    "C24_alkylamine_tetrahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_alkylamine_tetrahydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    "C24_alkylamine_pentahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_alkylamine_pentahydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',

    # "C28_alcohol_1OH": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C28_alcohol_1OH_SpecReBoot_filtered_MN_and_entropy.mgf',
    "C28_alcohol_2OH": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C28_alcohol_2OH_SpecReBoot_filtered_MN_and_entropy.mgf',
    "C28_alcohol_3OH": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C28_alcohol_3OH_SpecReBoot_filtered_MN_and_entropy.mgf',   
}

In [ ]:
#entropy
scans_post_entropy = pd.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/deltas_calculation/deltas_postclustering_entropy_remaining_scans.tsv', sep='\t')

# molecular networking
scans_post_MN = pd.read_csv(f'{PROJECT_DIR}/Scripts_data_analysis/molecular_networking_filter/All_with_SpecReBoot_and_remaining_scans.tsv', sep='\t')

def expand_scan_ids(scan_series):
    values = (
        scan_series.dropna()
        .astype(str)
        .str.split(';')
        .explode()
        .astype(str)
        .str.strip()
    )
    values = values[values != '']
    return values.unique().tolist()

def filter_mgf_by_scan_ids(mgf_input_path, mgf_output_path, scan_ids_to_keep):
    scan_ids_to_keep = {str(scan_id) for scan_id in scan_ids_to_keep}
    kept_spectra = 0
    total_spectra = 0

    with open(mgf_input_path, 'r') as fin, open(mgf_output_path, 'w') as fout:
        current_block = []
        current_scan = None
        in_block = False

        for line in fin:
            if line.startswith('BEGIN IONS'):
                in_block = True
                current_block = [line]
                current_scan = None
                continue

            if in_block:
                current_block.append(line)
                if line.startswith('SCANS='):
                    current_scan = line.strip().split('=', 1)[1]
                if line.startswith('END IONS'):
                    total_spectra += 1
                    if current_scan in scan_ids_to_keep:
                        fout.writelines(current_block)
                        kept_spectra += 1
                    in_block = False
                    current_block = []
            else:
                # Preserve any non-spectrum lines outside BEGIN/END IONS blocks.
                fout.write(line)

    return {
        'input_spectra': total_spectra,
        'kept_spectra': kept_spectra,
        'mgf_output': mgf_output_path,
    }

# For these groups, keep scans using entropy only (no MN intersection).
entropy_only_groups = {'C28_alcohol_1OH'}

def normalize_group_name(value):
    return str(value).strip().lower()

entropy_group_lookup = {normalize_group_name(name): name for name in scans_post_entropy['table_name'].dropna().astype(str)}
mn_group_lookup = {normalize_group_name(name): name for name in scans_post_MN['group'].dropna().astype(str)}
# Add aliasing for combined groups (e.g. 'c24_trihydroxy_combined' -> 'c24_trihydroxy')
entropy_group_lookup_extended = dict(entropy_group_lookup)
for k, v in list(entropy_group_lookup.items()):
    if k.endswith('_combined'):
        alias = k.replace('_combined', '')
        if alias not in entropy_group_lookup_extended:
            entropy_group_lookup_extended[alias] = v
entropy_groups_available = set(entropy_group_lookup_extended)
mn_groups_available = set(mn_group_lookup)
entropy_only_groups_normalized = {normalize_group_name(name) for name in entropy_only_groups}

# Use intersection for most groups, plus entropy-only groups that exist in the entropy table.
common_groups = sorted((entropy_groups_available & mn_groups_available) | (entropy_only_groups_normalized & entropy_groups_available))
if not common_groups:
    raise ValueError('No eligible groups were found between the entropy and molecular networking tables.')

final_results = []

for group_key in common_groups:
    group_name_entropy = entropy_group_lookup_extended[group_key]
    group_name_mn = mn_group_lookup.get(group_key, group_name_entropy)

    scans_post_entropy_filtered = scans_post_entropy[scans_post_entropy['table_name'] == group_name_entropy]
    scans_to_keep_entropy = expand_scan_ids(scans_post_entropy_filtered['remaining_scans_after_filters'])

    scans_post_MN_filtered = scans_post_MN[scans_post_MN['group'] == group_name_mn]
    scans_to_keep_MN = expand_scan_ids(scans_post_MN_filtered['remaining_scans_after_MN_filter'])

    if group_key in entropy_only_groups_normalized:
        final_scans_to_keep = sorted(
            set(scans_to_keep_entropy),
            key=lambda value: (not str(value).isdigit(), int(value) if str(value).isdigit() else str(value))
        )
        selection_method = 'entropy_only'
    else:
        final_scans_to_keep = sorted(
            set(scans_to_keep_entropy) & set(scans_to_keep_MN),
            key=lambda value: (not str(value).isdigit(), int(value) if str(value).isdigit() else str(value))
        )
        selection_method = 'entropy_and_MN_intersection'

    mgf_input_path = mgf_post_MN_filter.get(group_name_mn)
    mgf_output_path = mgf_post_entropy_and_MN.get(group_name_mn)

    if mgf_input_path is None or mgf_output_path is None:
        print(f'{group_name_mn}: skipped because the input or output MGF path is missing.')
        final_results.append({
            'group': group_name_mn,
            'selection_method': selection_method,
            'entropy_scans': len(scans_to_keep_entropy),
            'mn_scans': len(scans_to_keep_MN),
            'final_scans': len(final_scans_to_keep),
            'input_spectra': 'skipped',
            'kept_spectra': 'skipped',
            'mgf_output': 'skipped',
        })
        continue

    filter_result = filter_mgf_by_scan_ids(mgf_input_path, mgf_output_path, final_scans_to_keep)

    print(f'{group_name_mn} [{selection_method}]: entropy scans = {len(scans_to_keep_entropy)}, MN scans = {len(scans_to_keep_MN)}, final scans = {len(final_scans_to_keep)}')
    print(f"{group_name_mn}: kept {filter_result['kept_spectra']} of {filter_result['input_spectra']} spectra -> {mgf_output_path}")
    print('-' * 100)

    final_results.append({
        'group': group_name_mn,
        'selection_method': selection_method,
        'entropy_scans': len(scans_to_keep_entropy),
        'mn_scans': len(scans_to_keep_MN),
        'final_scans': len(final_scans_to_keep),
        'input_spectra': filter_result['input_spectra'],
        'kept_spectra': filter_result['kept_spectra'],
        'mgf_output': mgf_output_path,
    })

final_results_df = pd.DataFrame(final_results)
final_results_outfile = f'{PROJECT_DIR}/Scripts_data_analysis/deltas_calculation/results_entropy_and_MN_filter_with_specreboot.tsv'
final_results_df.to_csv(final_results_outfile, sep='\t', index=False)

print(f'Combined entropy + MN results exported to: {final_results_outfile}')
final_results_df

### Deltas calculation

In [ ]:
annotation_tables = {
    'c24_nonhydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_nonhydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    'c24_monohydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_monohydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    'c24_dihydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_dihydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    'c24_trihydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_trihydroxy_combined_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    'c24_tetrahydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_tetrahydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    'c24_pentahydroxy': f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_pentahydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',

    "C23_monohydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C23_monohydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    "C23_dihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C23_dihydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    "C23_trihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C23_trihydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    "C23_tetrahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C23_tetrahydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    "C23_pentahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C23_pentahydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',

    "C27_monohydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C27_monohydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    "C27_dihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C27_dihydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    "C27_trihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C27_trihydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    "C27_tetrahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C27_tetrahydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    "C27_pentahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C27_pentahydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',

    "Aceto_dihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/Aceto_dihydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',

    "C24_alkylamine_monohydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_alkylamine_monohydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    "C24_alkylamine_dihydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_alkylamine_dihydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    "C24_alkylamine_trihydroxy_combined": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_alkylamine_trihydroxy_combined_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    "C24_alkylamine_tetrahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_alkylamine_tetrahydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    "C24_alkylamine_pentahydroxy": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C24_alkylamine_pentahydroxy_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',

    "C28_alcohol_1OH": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C28_alcohol_1OH_falcon_eps01_minsamples2_scan_annotation_table.tsv',
    "C28_alcohol_2OH": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C28_alcohol_2OH_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',
    "C28_alcohol_3OH": f'{PROJECT_DIR}/Falcon/postMN_final_filter/final_mgfs/C28_alcohol_3OH_SpecReBoot_filtered_MN_and_entropy_annotation_table.tsv',  
}

deltas_calculation = {
    "c24_nonhydroxy": 361.3101,
    "c24_monohydroxy": 377.305,
    "c24_dihydroxy": 393.2999,
    "c24_trihydroxy": 409.2949,
    "c24_tetrahydroxy": 425.2898,
    "c24_pentahydroxy": 441.2847,

    "C23_monohydroxy": 363.2894,
    "C23_dihydroxy": 379.2843,
    "C23_trihydroxy": 395.2792,
    "C23_tetrahydroxy": 411.2741,
    "C23_pentahydroxy": 427.269,

    "C27_monohydroxy": 419.352,
    "C27_dihydroxy": 435.3469,
    "C27_trihydroxy": 451.3418,
    "C27_tetrahydroxy": 467.3367,
    "C27_pentahydroxy": 483.3316,

    "Aceto_dihydroxy": 435.3105,

    "C24_alkylamine_monohydroxy": 377.305,
    "C24_alkylamine_dihydroxy": 393.2999,
    "C24_alkylamine_trihydroxy_combined": 409.2949,
    "C24_alkylamine_tetrahydroxy": 425.2898,
    "C24_alkylamine_pentahydroxy": 441.2847,
}

In [ ]:
results = []

for group_name, file_path in annotation_tables.items():
    # Read the annotation table
    df_group = pd.read_csv(file_path, sep='\t')
    num_scans = len(df_group)
    
    # Try to find matching delta calculation key (handle case variations and '_combined' suffix)
    delta_key = None
    
    if group_name in deltas_calculation:
        delta_key = group_name
    else:
        # Try uppercase first (for lowercase c24_, c23_ groups)
        uppercase_name = group_name.upper() if group_name.startswith('c24_') else group_name
        if uppercase_name in deltas_calculation:
            delta_key = uppercase_name
        else:
            # Try replacing '_combined' 
            without_combined = group_name.replace('_combined', '')
            if without_combined in deltas_calculation:
                delta_key = without_combined
            else:
                # Try uppercase without combined
                uppercase_without_combined = without_combined.upper() if without_combined.startswith('c24_') else without_combined
                if uppercase_without_combined in deltas_calculation:
                    delta_key = uppercase_without_combined
    
    if delta_key:
        delta_mass = deltas_calculation[delta_key]
        df_group['delta'] = df_group['pepmass'] - delta_mass
        df_group['delta_rounded'] = df_group['delta'].round(2)
        num_deltas = df_group['delta_rounded'].nunique()
    else:
        # C28 alcohols or other groups without delta calculation
        num_deltas = "unspecified"
    
    results.append({
        'group': group_name,
        'num_scans': num_scans,
        'num_deltas': num_deltas
    })

# Create DataFrame and export
results_df = pd.DataFrame(results)
output_file = f'{PROJECT_DIR}/Scripts_data_analysis/deltas_calculation/deltas_summary_postMN_specreboot_post_entropy.tsv'
results_df.to_csv(output_file, sep='\t', index=False)

print(f"Summary exported to: {output_file}")
print(results_df)